In [1]:
# ===== CELL 1 — clone + start clock =====
import time, subprocess
NB_START = time.time()
subprocess.run("mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && "
               "git clone -q https://github.com/CIawevy/FreeFine.git", shell=True, check=True)
print("cloned; clock started")

cloned; clock started


In [2]:
%%bash
# ===== CELL 2 — freefine_env (generation) =====
set -e
pip install -q --root-user-action=ignore uv
uv python install 3.10.13
V=/kaggle/temp/freefine_env; PY=$V/bin/python
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" --index-url https://download.pytorch.org/whl/cu121
cd /kaggle/temp/FreeFine
uv pip install --python "$PY" -r requirements.txt || { grep -v '^xformers' requirements.txt>/tmp/r.txt; uv pip install --python "$PY" -r /tmp/r.txt; uv pip install --python "$PY" xformers; }
uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70"
"$PY" -c "import torch,diffusers,xformers; print('freefine_env OK',torch.__version__,diffusers.__version__)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 72.5 MB/s eta 0:00:00
freefine_env OK 2.1.1+cu121 0.18.0


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.47s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/freefine_env
Activate with: source /kaggle/temp/freefine_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/freefine_env
Resolved 18 packages in 1.56s
 Downloaded torchvision
 Downloaded triton
 Downloaded pillow
 Downloaded networkx
 Downloaded numpy
 Downloaded sympy
 Downloaded torch
Prepared 18 packages in 46.05s
Installed 18 packages in 290ms
 + certifi==2022.12.7
 + charset-normalizer==2.1.1
 + filelock==3.29.0
 + fsspec==2026.4.0
 + idna==3.4
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + pillow==12.2.0
 + requests==2.28.1
 + sympy==1.14.0
 + torch==2.1.1+cu121
 + torchvision==0.16.1+cu121
 + triton==2.1.0
 + typing-extensions==4.15.0
 + urllib3==1.26.13
Using Python 3.10.13 environment at: /kaggle/temp/freefine_en

In [3]:
%%bash
# ===== CELL 3 — metric_env (evaluation) =====
set -e
V=/kaggle/temp/metric_env; PY=$V/bin/python; REPO=/kaggle/temp/FreeFine
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null||true; done
echo "metric_env OK -> datasets $($PY -c 'import datasets;print(datasets.__version__)')"

metric_env OK -> datasets 2.21.0


Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 900ms
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded torchaudio
 Downloaded torchvision
 Downloaded nvidia-nvjitlink-cu12
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded pillow
 Downloaded nvidia-curand-cu12
 Downloaded networkx
 Downloaded triton
 Downloaded nvidia-cusolver-cu12
 Downloaded numpy
 Downloaded nvidia-cusparselt-cu12
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded sympy
 Downloaded nvidia-cublas-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 27 packages in 46.83s
Installed 27 packages in 399ms
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.4.5.8
 + nvidia-cuda-cupti-cu12==12.4

In [4]:
# ===== CELL 4 — patch metric code (args.3d, SD-2.1 mirror, SEEDED MD) =====
import pathlib
root=pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp=root/"main.py"; mp.write_text(mp.read_text().replace("args.3d","getattr(args,'3d')"))
for f in [root/"MD"/"mean_distance.py", root/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace("stabilityai/stable-diffusion-2-1","sd2-community/stable-diffusion-2-1"))
md=root/"MD"/"mean_distance.py"; s=md.read_text(); assert "all_dist = []" in s
s=s.replace("all_dist = []","all_dist = []\n    import torch as _st; _st.manual_seed(42); _st.cuda.manual_seed_all(42)",1)
md.write_text(s); print("metric patched (seeded MD)")

metric patched (seeded MD)


In [5]:
# ===== CELL 5 (fixed) — pick the right cache + verify symlinks =====
import os, glob, json, csv, random, shutil
from collections import defaultdict, Counter
GEO="/kaggle/temp/GeoBenchMeta"; os.makedirs(f"{GEO}/Geo-Bench-2D",exist_ok=True)
cands=glob.glob("/kaggle/input/**/Geo-Bench-2D",recursive=True)
CACHE=next((c for c in cands if os.path.isdir(f"{c}/source_img")), None)
assert CACHE, f"no Geo-Bench-2D with source_img among {cands}"
print("CACHE:",CACHE)
ANN=glob.glob("/kaggle/input/**/annotation_2d.json",recursive=True)[0]
IB=os.path.dirname(os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/inp_img_blended/**/inp_img.png",recursive=True)[0])))
META=glob.glob("/kaggle/input/**/sample_metadata.csv",recursive=True)[0]
for nm in ["source_img","source_mask","target_mask","source_img_full_v2"]:
    d=f"{GEO}/Geo-Bench-2D/{nm}"
    if os.path.islink(d): os.remove(d)
    elif os.path.isdir(d): shutil.rmtree(d)
    os.symlink(f"{CACHE}/{nm}", d)
d=f"{GEO}/Geo-Bench-2D/inp_img_blended"
if os.path.islink(d): os.remove(d)
os.symlink(IB, d)
nsrc=len(glob.glob(f"{GEO}/Geo-Bench-2D/source_img/*.png"))
assert nsrc>=600, f"source_img only has {nsrc} files — wrong cache"
print(f"symlinks verified: source_img has {nsrc} files")
shutil.copy(ANN,f"{GEO}/annotation_2d.json"); ann=json.load(open(f"{GEO}/annotation_2d.json"))
meta=[r for r in csv.DictReader(open(META)) if os.path.exists(f"{IB}/{r['da_n']}/{r['ins_id']}/inp_img.png")]
random.seed(42); cells=defaultdict(list)
for r in meta: cells[(r["edit_type"],r["difficulty"])].append(r)
keys=sorted(cells); per=200//len(keys); picked=[]
for k in keys:
    pool=cells[k][:]; random.shuffle(pool); picked+=pool[:per]
chosen={(r["da_n"],r["ins_id"],r["case_id"]) for r in picked}
left=[r for r in meta if (r["da_n"],r["ins_id"],r["case_id"]) not in chosen]; random.shuffle(left)
for r in left:
    if len(picked)>=200: break
    picked.append(r)
picked=picked[:200]
print("subset",len(picked),dict(Counter(r["edit_type"] for r in picked)),dict(Counter(r["difficulty"] for r in picked)))
json.dump(picked,open(f"{GEO}/subset_meta.json","w"))
gsub={}
for r in picked:
    d,i,e=r["da_n"],r["ins_id"],r["case_id"]; lf=dict(ann[d]["instances"][i][e])
    lf["ori_img_path"]=os.path.join(GEO,lf["ori_img_path"]); lf["ori_mask_path"]=os.path.join(GEO,lf["ori_mask_path"])
    gsub.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
json.dump(gsub,open(f"{GEO}/gen_subset.json","w")); print("wrote gen_subset.json")

CACHE: /kaggle/input/geobench2d-metrics-subset/geobench_metrics/Geo-Bench-2D
symlinks verified: source_img has 609 files
subset 200 {'move': 67, 'resize': 67, 'rotate': 66} {'easy': 67, 'hard': 67, 'medium': 66}
wrote gen_subset.json


In [6]:
# ===== CELL 5b — make coarse_img available (prefer attached dataset; else download 200) =====
import os, json, glob, time
GEO="/kaggle/temp/GeoBenchMeta"; cdst=f"{GEO}/Geo-Bench-2D/coarse_img"
hits=glob.glob("/kaggle/input/**/coarse_img/*/*/*.png", recursive=True)
if hits:
    CIMG=hits[0].split("/coarse_img/")[0]+"/coarse_img"      # the attached coarse_img dir
    if not (os.path.islink(cdst) or os.path.exists(cdst)): os.symlink(CIMG, cdst)
    print("using ATTACHED coarse_img ->", CIMG, "| files:", len(glob.glob(cdst+"/**/*.png",recursive=True)))
else:
    from huggingface_hub import snapshot_download
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"]=UserSecretsClient().get_secret("HF_TOKEN")
    ann=json.load(open(f"{GEO}/annotation_2d.json")); picked=json.load(open(f"{GEO}/subset_meta.json"))
    pats=sorted({ann[r["da_n"]]["instances"][r["ins_id"]][r["case_id"]]["coarse_input_path"] for r in picked})
    DL="/kaggle/temp/coarse_dl"
    for a in range(6):
        try: snapshot_download("CIawevy/GeoBenchMeta",repo_type="dataset",local_dir=DL,token=os.environ["HF_TOKEN"],max_workers=4,allow_patterns=pats); break
        except Exception as e: print("retry coarse:",str(e)[:90],flush=True); time.sleep(45)
    if not os.path.exists(cdst): os.symlink(f"{DL}/Geo-Bench-2D/coarse_img", cdst)
    print("DOWNLOADED coarse ->", cdst, "| files:", len(glob.glob(cdst+"/**/*.png",recursive=True)))

# hard gate: every subset case must have its coarse resolvable (else WRAP_E would crash)
ann=json.load(open(f"{GEO}/annotation_2d.json")); picked=json.load(open(f"{GEO}/subset_meta.json"))
miss=[r for r in picked if not os.path.exists(os.path.join(GEO, ann[r["da_n"]]["instances"][r["ins_id"]][r["case_id"]]["coarse_input_path"]))]
print("subset coarse missing:", len(miss))
assert len(miss)==0, f"{len(miss)} subset cases lack coarse — WRAP_E would fail"

using ATTACHED coarse_img -> /kaggle/input/datasets/georgiostzamouranis/geobench2d-coarse-img/geobench_coarse/Geo-Bench-2D/coarse_img | files: 5677
subset coarse missing: 0


In [7]:
# ===== CELL 6 — parametrize generation script =====
import os
P="/kaggle/temp/FreeFine/evaluation/FreeFine"; src=open(f"{P}/freefine_batch_infer_2d.py").read()
src=src.replace("sys.path.append('/data/Hszhu/FreeFine')","sys.path.append('/kaggle/temp/FreeFine')")
src=src.replace('pretrained_model_path = "/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/"',
                'pretrained_model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"')
old=('        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n'
     '        obj_label = ""\n        ori_mask = read_and_resize_mask(ori_mask_path)\n')
new=('        ori_mask = read_and_resize_mask(ori_mask_path)\n'
     '        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n        obj_label = ""\n')
assert old in src,"ori_mask block mismatch"; src=src.replace(old,new,1)
src=src.replace("controller = Attention_Modulator(start_layer=10)",
                "controller = Attention_Modulator(start_layer=int(os.environ.get('FF_START_LAYER','10')))")
src=src.replace('"guidance_scale": 7.5,','"guidance_scale": float(os.environ.get("FF_GUIDANCE","7.5")),')
src=src.replace('"start_step": 35,','"start_step": int(os.environ.get("FF_START_STEP","35")),')
src=src.replace('dataset_json = osp.join(dst_base, "annotations_2d.json")','dataset_json = os.environ.get("FF_SUBSET_JSON", osp.join(dst_base,"annotations_2d.json"))')
src=src.replace('dst_gen_dir = osp.join(dst_base, "Geo-Bench-2D/Gen_results_FreeFine_2d")','dst_gen_dir = os.environ.get("FF_OUT_DIR", osp.join(dst_base,"Geo-Bench-2D/Gen_results_FreeFine_2d"))')
src=src.replace('base_dir = "/data/Hszhu/dataset/GeoBenchMeta/"','base_dir = "/kaggle/temp/GeoBenchMeta"')
open(f"{P}/freefine_sweep_2d.py","w").write(src); print("parametrized")

parametrized


In [8]:
# ===== CELL 7 (final gate) =====
import os, json, socket, subprocess, glob, numpy as np
from PIL import Image
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"]=UserSecretsClient().get_secret("HF_TOKEN")
GEO="/kaggle/temp/GeoBenchMeta"; P="/kaggle/temp/FreeFine/evaluation/FreeFine"
GENBASE=glob.glob('/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup',recursive=True)[0]
gs=json.load(open(f"{GEO}/gen_subset.json")); val={}; n=0
for d,da in gs.items():
    for i,ins in da["instances"].items():
        for e in ins:
            val.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=ins[e]; n+=1
            if n>=5: break
        if n>=5: break
    if n>=5: break
json.dump(val,open(f"{GEO}/val5.json","w")); os.makedirs("/kaggle/temp/val5",exist_ok=True)
env=os.environ.copy(); env.update({"PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],"PYTHONUNBUFFERED":"1",
    "TOKENIZERS_PARALLELISM":"false","HF_HOME":"/kaggle/temp/hf","NCCL_P2P_DISABLE":"1",
    "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","FF_SUBSET_JSON":f"{GEO}/val5.json","FF_OUT_DIR":"/kaggle/temp/val5"})
s=socket.socket();s.bind(("",0));port=s.getsockname()[1];s.close()
r=subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=1","--master-port",str(port),"freefine_sweep_2d.py"],
    cwd=P,env=env,capture_output=True,text=True)
imgs=sorted(glob.glob("/kaggle/temp/val5/**/*.png",recursive=True))
if not imgs: print((r.stdout+r.stderr)[-3000:]); raise SystemExit("validation produced 0 images — abort")
diffs=[]
for nimg in imgs:
    rel=os.path.relpath(nimg,"/kaggle/temp/val5")
    A=np.array(Image.open(nimg).convert("RGB")).astype(float)
    B=np.array(Image.open(f"{GENBASE}/{rel}").convert("RGB").resize(Image.open(nimg).size)).astype(float)
    diffs.append(float(np.abs(A-B).mean()))
print("validation mean|Δ|:",[round(x,3) for x in diffs])
assert max(diffs)<1.0, f"PIPELINE NOT REPRODUCING (max Δ={max(diffs):.2f}) — ABORT"
print("✓ validation passed — full sweep cleared")

validation mean|Δ|: [0.0, 0.0, 0.0, 0.0, 0.0]
✓ validation passed — full sweep cleared


In [9]:
# ===== CELL 8 — FULL SWEEP (time-boxed; saves each variant) =====
import os, time, glob, subprocess, socket
GEO="/kaggle/temp/GeoBenchMeta"; OUTROOT="/kaggle/working/phase2/variants"; os.makedirs(OUTROOT,exist_ok=True)
GEN_DEADLINE=NB_START+8.5*3600   # stop launching new variants after 8.5h (12h-cliff safety)
VARIANTS=[("ss25",{"FF_START_STEP":25}),("ss30",{"FF_START_STEP":30}),
          ("ss40",{"FF_START_STEP":40}),("ss45",{"FF_START_STEP":45}),
          ("g5",{"FF_GUIDANCE":5.0}),("g10",{"FF_GUIDANCE":10.0}),
          ("l8",{"FF_START_LAYER":8}),("l12",{"FF_START_LAYER":12})]
def gen2(outdir,**kw):
    os.makedirs(outdir,exist_ok=True); env=os.environ.copy()
    env.update({"PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],"PYTHONUNBUFFERED":"1",
        "TOKENIZERS_PARALLELISM":"false","HF_HOME":"/kaggle/temp/hf","NCCL_P2P_DISABLE":"1",
        "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","FF_SUBSET_JSON":f"{GEO}/gen_subset.json","FF_OUT_DIR":outdir})
    env.update({k:str(v) for k,v in kw.items()})
    s=socket.socket();s.bind(("",0));port=s.getsockname()[1];s.close()
    return subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=2","--master-port",str(port),"freefine_sweep_2d.py"],
        cwd="/kaggle/temp/FreeFine/evaluation/FreeFine",env=env,stdout=open(outdir+"/log.txt","w"),stderr=subprocess.STDOUT)
done=[]
for tag,kw in VARIANTS:
    if time.time()>GEN_DEADLINE: print(f"[{int((time.time()-NB_START)/60)}m] deadline — skip {tag}+rest",flush=True); break
    out=f"{OUTROOT}/{tag}"; t=time.time()
    try:
        r=gen2(out,**kw); k=len(glob.glob(out+"/**/*.png",recursive=True))
        print(f"[{int((time.time()-NB_START)/60)}m] {tag}: rc={r.returncode} {int(time.time()-t)}s imgs={k}",flush=True)
        if k>0: done.append(tag)
        if r.returncode!=0: print("  tail:",open(out+'/log.txt').read()[-900:])
    except Exception as ex: print(f"{tag} FAILED:{ex}",flush=True)
print("variants generated:",done,flush=True)

[121m] ss25: rc=0 6638s imgs=200
[210m] ss30: rc=0 5362s imgs=200
[257m] ss40: rc=0 2794s imgs=200
[282m] ss45: rc=0 1508s imgs=200
[350m] g5: rc=0 4081s imgs=200
[418m] g10: rc=0 4081s imgs=200
[486m] l8: rc=0 4081s imgs=200
[554m] l12: rc=0 4081s imgs=200
variants generated: ['ss25', 'ss30', 'ss40', 'ss45', 'g5', 'g10', 'l8', 'l12']


In [10]:
# ===== CELL 9 — EVAL (SUBC all groups, seeded MD on hard cells; time-boxed) =====
import os, json, re, glob, subprocess, time
GEO="/kaggle/temp/GeoBenchMeta"; MET="/kaggle/temp/FreeFine/evaluation/metrics"; PY="/kaggle/temp/metric_env/bin/python"
EVAL_DEADLINE=NB_START+10.5*3600
ann=json.load(open(f"{GEO}/annotation_2d.json")); picked=json.load(open(f"{GEO}/subset_meta.json"))
GENBASE=glob.glob('/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup',recursive=True)[0]
os.makedirs(f"{GEO}/gen_eval",exist_ok=True)
sets={"baseline":GENBASE}
for tag in ["ss25","ss30","ss40","ss45","g5","g10","l8","l12"]:
    p=f"/kaggle/working/phase2/variants/{tag}"
    if glob.glob(p+"/**/*.png",recursive=True): sets[tag]=p
for nm,srcp in sets.items():
    d=f"{GEO}/gen_eval/{nm}"
    if os.path.islink(d): os.remove(d)
    os.symlink(srcp,d)
def members(pred): return [(r["da_n"],r["ins_id"],r["case_id"]) for r in picked if pred(r)]
groups={"rotate_hard":members(lambda r:r["edit_type"]=="rotate" and r["difficulty"]=="hard"),
        "resize_hard":members(lambda r:r["edit_type"]=="resize" and r["difficulty"]=="hard"),
        "move_all":members(lambda r:r["edit_type"]=="move"),"all_200":members(lambda r:True)}
md_groups={"rotate_hard","resize_hard"}
def manifest(setname,ids):
    o={}; base=f"{GEO}/gen_eval/{setname}"
    for d,i,e in ids:
        if not os.path.exists(f"{base}/{d}/{i}/{e}.png"): continue
        lf=dict(ann[d]["instances"][i][e]); lf["gen_img_path"]=f"gen_eval/{setname}/{d}/{i}/{e}.png"
        o.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
    pth=f"{GEO}/man_{setname}_{len(ids)}.json"; json.dump(o,open(pth,"w"))
    return pth,sum(len(c) for da in o.values() for c in da["instances"].values())
def run_metric(manp,task):
    env=os.environ.copy(); env.update({"MPLBACKEND":"Agg","PYTHONUNBUFFERED":"1","HF_HOME":"/kaggle/temp/hf","TORCH_HOME":"/kaggle/temp/torch","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True"})
    out=subprocess.run([PY,"main.py","--path",manp,"--use_relative_path","--base_dir",GEO,"--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2","--task",task,"--level","0"],cwd=MET,env=env,capture_output=True,text=True)
    t=out.stdout+out.stderr; v={}
    for k in ["SUBC","MD"]:
        m=re.findall(rf"{k}:\s*([-\d.eE]+)",t)
        if m: v[k]=round(float(m[-1]),4)
    return v,t
results={}; stop=False
for setname in sets:
    if stop: break
    results[setname]={}
    for gname,ids in groups.items():
        if time.time()>EVAL_DEADLINE: print("EVAL_DEADLINE — stopping",flush=True); stop=True; break
        if not ids: continue
        man,cnt=manifest(setname,ids)
        if cnt==0: continue
        task="000010100" if gname in md_groups else "000010000"
        v,t=run_metric(man,task); v["n"]=cnt; results[setname][gname]=v
        print(f"{setname:9s} {gname:12s} n={cnt:3d} -> {v}",flush=True)
        if not v: print("  parse-fail:",t[-700:])
    json.dump(results,open("/kaggle/working/phase2/results.json","w"),indent=2)
print("eval done")

baseline  rotate_hard  n= 22 -> {'SUBC': 0.8523, 'MD': 14.0096, 'n': 22}
baseline  resize_hard  n= 23 -> {'SUBC': 0.8391, 'MD': 19.4713, 'n': 23}
baseline  move_all     n= 67 -> {'SUBC': 0.959, 'n': 67}
baseline  all_200      n=200 -> {'SUBC': 0.9154, 'n': 200}
ss25      rotate_hard  n= 22 -> {'SUBC': 0.8497, 'MD': 15.2464, 'n': 22}
ss25      resize_hard  n= 23 -> {'SUBC': 0.8319, 'MD': 21.1346, 'n': 23}
ss25      move_all     n= 67 -> {'SUBC': 0.9542, 'n': 67}
ss25      all_200      n=200 -> {'SUBC': 0.9113, 'n': 200}
ss30      rotate_hard  n= 22 -> {'SUBC': 0.8519, 'MD': 15.9111, 'n': 22}
ss30      resize_hard  n= 23 -> {'SUBC': 0.8376, 'MD': 20.7004, 'n': 23}
ss30      move_all     n= 67 -> {'SUBC': 0.9571, 'n': 67}
ss30      all_200      n=200 -> {'SUBC': 0.9139, 'n': 200}
ss40      rotate_hard  n= 22 -> {'SUBC': 0.8528, 'MD': 15.0132, 'n': 22}
ss40      resize_hard  n= 23 -> {'SUBC': 0.8413, 'MD': 20.1565, 'n': 23}
ss40      move_all     n= 67 -> {'SUBC': 0.9605, 'n': 67}
ss40    

In [11]:
# ===== CELL 10 — summary table =====
import json
R=json.load(open("/kaggle/working/phase2/results.json"))
order=["baseline","ss25","ss30","ss40","ss45","g5","g10","l8","l12"]
print("PHASE-2 SWEEP — balanced-200  (baseline = start_step35/cfg7.5/layer10)\n")
for grp in ["rotate_hard","resize_hard","move_all","all_200"]:
    print(f"=== {grp} ===");  print(f"{'set':10s}{'n':>5}{'SUBC↑':>9}{'WRAP_E↓':>10}{'MD↓':>9}")
    for s in order:
        if s in R and grp in R[s]:
            d=R[s][grp]
            print(f"{s:10s}{d.get('n','-'):>5}{str(d.get('SUBC','-')):>9}{str(d.get('WRAP_E','-')):>10}{str(d.get('MD','-')):>9}")
    print()
print("saved -> /kaggle/working/phase2/results.json + variants/")

PHASE-2 SWEEP — balanced-200  (baseline = start_step35/cfg7.5/layer10)

=== rotate_hard ===
set           n    SUBC↑   WRAP_E↓      MD↓
baseline     22   0.8523         -  14.0096
ss25         22   0.8497         -  15.2464
ss30         22   0.8519         -  15.9111
ss40         22   0.8528         -  15.0132
ss45         22   0.8529         -  15.0901
g5           22   0.8523         -  14.0096
g10          22   0.8523         -  14.0096
l8           22   0.8523         -  14.0096
l12          22   0.8523         -  14.0096

=== resize_hard ===
set           n    SUBC↑   WRAP_E↓      MD↓
baseline     23   0.8391         -  19.4713
ss25         23   0.8319         -  21.1346
ss30         23   0.8376         -  20.7004
ss40         23   0.8413         -  20.1565
ss45         23   0.8414         -  18.0894
g5           23   0.8391         -  19.4713
g10          23   0.8391         -  19.4713
l8           23   0.8391         -  19.4713
l12          23   0.8391         -  19.4713

=== mo